In [5]:
pip install pyspark


Note: you may need to restart the kernel to use updated packages.


In [6]:
from pyspark.sql.functions import col
from pyspark.sql.types import IntegerType, DoubleType, BooleanType, DateType

In [7]:
configs = {"fs.azure.account.auth.type": "OAuth",
"fs.azure.account.oauth.provider.type": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
"fs.azure.account.oauth2.client.id": "",
"fs.azure.account.oauth2.client.secret": '',
"fs.azure.account.oauth2.client.endpoint": "https://login.microsoftonline.com/tanent_id/oauth2/token"}


dbutils.fs.mount(
source = "abfss://tokyo-olympic-data@tokyoolympicdata.dfs.core.windows.net", # contrainer@storageacc
mount_point = "/mnt/tokyoolymic",
extra_configs = configs)
  

NameError: name 'dbutils' is not defined

In [ ]:
%fs
ls "/mnt/tokyoolymic"

In [ ]:
spark

In [ ]:
athletes = spark.read.format("csv").option("header","true").option("inferSchema","true").load("/mnt/tokyoolymic/raw-data/athletes.csv")
coaches = spark.read.format("csv").option("header","true").option("inferSchema","true").load("/mnt/tokyoolymic/raw-data/coaches.csv")
entriesgender = spark.read.format("csv").option("header","true").option("inferSchema","true").load("/mnt/tokyoolymic/raw-data/entriesgender.csv")
medals = spark.read.format("csv").option("header","true").option("inferSchema","true").load("/mnt/tokyoolymic/raw-data/medals.csv")
teams = spark.read.format("csv").option("header","true").option("inferSchema","true").load("/mnt/tokyoolymic/raw-data/teams.csv")

In [ ]:
athletes.show()

In [ ]:
athletes.printSchema()

In [ ]:
coaches.show()

In [ ]:
coaches.printSchema()

In [ ]:
entriesgender.show()

In [ ]:
entriesgender.printSchema()

In [ ]:
entriesgender = entriesgender.withColumn("Female",col("Female").cast(IntegerType()))\
    .withColumn("Male",col("Male").cast(IntegerType()))\
    .withColumn("Total",col("Total").cast(IntegerType()))

In [ ]:
entriesgender.printSchema()

In [ ]:
medals.show()

In [ ]:
medals.printSchema()

In [ ]:
teams.show()

In [ ]:
teams.printSchema()

In [ ]:
# Find the top countries with the highest number of gold medals
top_gold_medal_countries = medals.orderBy("Gold", ascending=False).select("Team_Country","Gold").show()

In [ ]:
# Calculate the average number of entries by gender for each discipline
average_entries_by_gender = entriesgender.withColumn(
    'Avg_Female', entriesgender['Female'] / entriesgender['Total']
).withColumn(
    'Avg_Male', entriesgender['Male'] / entriesgender['Total']
)
average_entries_by_gender.show()

In [ ]:
athletes.repartition(1).write.mode("overwrite").option("header",'true').csv("/mnt/tokyoolymic/transformed-data/athletes")

In [ ]:
coaches.repartition(1).write.mode("overwrite").option("header","true").csv("/mnt/tokyoolymic/transformed-data/coaches")
entriesgender.repartition(1).write.mode("overwrite").option("header","true").csv("/mnt/tokyoolymic/transformed-data/entriesgender")
medals.repartition(1).write.mode("overwrite").option("header","true").csv("/mnt/tokyoolymic/transformed-data/medals")
teams.repartition(1).write.mode("overwrite").option("header","true").csv("/mnt/tokyoolymic/transformed-data/teams")

## Data Cleaning and Transformation

In [ ]:
athletes = athletes.dropDuplicates()
coaches = coaches.dropDuplicates()
entriesgender = entriesgender.dropDuplicates()
medals = medals.dropDuplicates()
teams = teams.dropDuplicates()


In [8]:
# Filling null values with 'Unknown' for categorical columns
athletes = athletes.fillna({'Country': 'Unknown', 'Discipline': 'Unknown'})
coaches = coaches.fillna({'Country': 'Unknown', 'Discipline': 'Unknown', 'Event': 'Unknown'})
teams = teams.fillna({'Country': 'Unknown', 'Discipline': 'Unknown', 'Event': 'Unknown'})

# Filling numerical columns with 0
entriesgender = entriesgender.fillna({'Female': 0, 'Male': 0, 'Total': 0})
medals = medals.fillna({'Gold': 0, 'Silver': 0, 'Bronze': 0, 'Total': 0})


NameError: name 'athletes' is not defined

 Standardize Country Names

In [ ]:
from pyspark.sql.functions import trim, upper

athletes = athletes.withColumn("Country", trim(upper(col("Country"))))
coaches = coaches.withColumn("Country", trim(upper(col("Country"))))
teams = teams.withColumn("Country", trim(upper(col("Country"))))
medals = medals.withColumn("Team_Country", trim(upper(col("Team_Country"))))


Find the Country with the Most Medals

In [ ]:
most_medals_country = medals.orderBy(col("Total").desc()).select("Team_Country", "Total").limit(1)
most_medals_country.show()


 Find the Discipline with the Highest Participation

In [9]:
most_participated_discipline = entriesgender.orderBy(col("Total").desc()).select("Discipline", "Total").limit(1)
most_participated_discipline.show()


NameError: name 'entriesgender' is not defined

Save Transformed Data Back to Azure Data Lake

In [ ]:
athletes.write.mode("overwrite").parquet("/mnt/tokyoolymic/transformed-data/athletes")
coaches.write.mode("overwrite").parquet("/mnt/tokyoolymic/transformed-data/coaches")
entriesgender.write.mode("overwrite").parquet("/mnt/tokyoolymic/transformed-data/entriesgender")
medals.write.mode("overwrite").parquet("/mnt/tokyoolymic/transformed-data/medals")
teams.write.mode("overwrite").parquet("/mnt/tokyoolymic/transformed-data/teams")
